# Phase 2 — Few-Shot Prototype Classification
11. Prototype construction (mean embedding per class) for 1/3/5-shot
12. Nearest-prototype classification
13. 5 seeds per shot-setting -> mean +/- std (avoids reporting a lucky/unlucky draw)
14. Compare 1/3/5-shot vs Phase 1 linear baseline

**Input:** `eyeclip_embeddings.npz` produced by the Phase 1 notebook (X_train/y_train/X_test/y_test).
No re-extraction, no model loading here — this phase only touches embeddings already on disk.

**Assumption:** support examples for prototypes are drawn from `X_train` (not `X_val`/`X_test`) — val/test stay untouched for evaluation, matching how few-shot is evaluated everywhere else in the plan.

In [7]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

EMB_PATH = r"E:\rop\codes\eyeclip_embeddings.zip"   # CHANGE if stored elsewhere
SHOTS = [1, 3, 5]
SEEDS = [0, 1, 2, 3, 4]
DISTANCE_METRIC = "cosine"            # "cosine" or "euclidean"

data = np.load(EMB_PATH)
X_train, y_train = data["X_train"], data["y_train"]
X_test,  y_test  = data["X_test"],  data["y_test"]
classes = sorted(np.unique(y_train))

print("train:", X_train.shape, "test:", X_test.shape)
print("classes:", classes)

# sanity: every class must have at least max(SHOTS) training examples,
# else k-shot sampling for that class will silently under-sample.
counts = pd.Series(y_train).value_counts().sort_index()
print("\ntrain examples per class:")
print(counts)
min_needed = max(SHOTS)
short_classes = counts[counts < min_needed]
if len(short_classes) > 0:
    print(f"\nWARNING: classes with fewer than {min_needed} train examples "
          f"(will use all available, not a full {min_needed}-shot):")
    print(short_classes)

train: (1411, 512) test: (320, 512)
classes: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]

train examples per class:
0    100
1    240
2    168
3    243
4    166
5     40
6     20
7    434
Name: count, dtype: int64


## Prototype construction + nearest-prototype classification

**Explainability:**
- A *prototype* is just the mean embedding vector of the k support examples sampled for a class.
- Classification = for each test embedding, find which class's prototype is closest.
- Cosine distance is used by default because embedding *direction* (not magnitude) is what CLIP-style encoders are trained to make meaningful.

In [8]:
def build_prototypes(X, y, classes, k, rng):
    """Sample k examples per class from (X, y) and average them.
    Input:  X [N,512], y [N], classes list, k int, rng np.random.Generator
    Output: prototypes [C,512] float32, in the same order as `classes`
    """
    prototypes = []
    for c in classes:
        idx = np.where(y == c)[0]
        n_take = min(k, len(idx))  # guard: class with < k examples uses all it has
        chosen = rng.choice(idx, size=n_take, replace=False)
        prototypes.append(X[chosen].mean(axis=0))
    return np.stack(prototypes)


def nearest_prototype_predict(X, prototypes, classes, metric="cosine"):
    """Assign each row of X to its nearest prototype.
    Input:  X [N,512], prototypes [C,512]
    Output: preds [N] using original class labels (not 0..C-1 indices)
    """
    if metric == "cosine":
        X_n = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-8)
        P_n = prototypes / (np.linalg.norm(prototypes, axis=1, keepdims=True) + 1e-8)
        sim = X_n @ P_n.T          # [N, C], higher = closer
        nearest_idx = sim.argmax(axis=1)
    elif metric == "euclidean":
        dists = np.linalg.norm(X[:, None, :] - prototypes[None, :, :], axis=2)  # [N, C]
        nearest_idx = dists.argmin(axis=1)
    else:
        raise ValueError(f"unknown metric: {metric}")
    return np.array([classes[i] for i in nearest_idx])

In [9]:
# self-check before trusting results (rubber-duck):
# - are seeds actually changing which support examples get picked? verify below.
rng_check = np.random.default_rng(SEEDS[0])
proto_a = build_prototypes(X_train, y_train, classes, k=1, rng=rng_check)
rng_check2 = np.random.default_rng(SEEDS[1])
proto_b = build_prototypes(X_train, y_train, classes, k=1, rng=rng_check2)
assert not np.allclose(proto_a, proto_b), "seeds are not producing different support sets"
print("seed variation confirmed: different seeds -> different 1-shot prototypes")

seed variation confirmed: different seeds -> different 1-shot prototypes


## Run all (shot, seed) combinations

In [10]:
results = []  # one row per (shot, seed)

for k in SHOTS:
    for seed in SEEDS:
        rng = np.random.default_rng(seed)
        prototypes = build_prototypes(X_train, y_train, classes, k, rng)
        preds = nearest_prototype_predict(X_test, prototypes, classes, DISTANCE_METRIC)

        acc = accuracy_score(y_test, preds)
        _, _, f1_per_class, _ = precision_recall_fscore_support(
            y_test, preds, labels=classes, average=None, zero_division=0)
        macro_f1 = f1_per_class.mean()

        results.append({"shot": k, "seed": seed, "accuracy": acc, "macro_f1": macro_f1})

results_df = pd.DataFrame(results)
results_df

,shot,seed,accuracy,macro_f1
0,1,0,0.137500,0.124073
1,1,1,0.153125,0.137527
2,1,2,0.156250,0.113267
3,1,3,0.156250,0.113924
4,1,4,0.168750,0.147512
5,3,0,0.175000,0.142213
6,3,1,0.156250,0.137929
7,3,2,0.071875,0.053946
8,3,3,0.143750,0.092773
9,3,4,0.150000,0.118481


## Aggregate: mean +/- std per shot-setting
This is the number that survives across seeds -- never report a single run.

In [ ]:
summary = results_df.groupby("shot").agg(
    accuracy_mean=("accuracy", "mean"),
    accuracy_std=("accuracy", "std"),
    macro_f1_mean=("macro_f1", "mean"),
    macro_f1_std=("macro_f1", "std"),
).reset_index()

# Phase 1 baseline for direct comparison -- fill in your actual Phase 1 numbers
PHASE1_ACCURACY = 0.3063
PHASE1_MACRO_F1 = 0.2469
baseline_row = pd.DataFrame([{
    "shot": "Phase 1 baseline (linear, full train set)",
    "accuracy_mean": PHASE1_ACCURACY, "accuracy_std": 0.0,
    "macro_f1_mean": PHASE1_MACRO_F1, "macro_f1_std": 0.0,
}])

final_table = pd.concat([summary, baseline_row], ignore_index=True)
print(final_table.to_string(index=False))
final_table.to_csv("phase2_fewshot_results.csv", index=False)

                                     shot  accuracy_mean  accuracy_std  macro_f1_mean  macro_f1_std
                                        1       0.154375      0.011180       0.127261      0.014995
                                        3       0.139375      0.039504       0.109069      0.036483
                                        5       0.133125      0.029031       0.097823      0.021910
Phase 1 baseline (linear, full train set)       0.306300      0.000000       0.246900      0.000000


: 

## Interpretation notes
- Compare `accuracy_mean` / `macro_f1_mean` across 1 -> 3 -> 5 shot: expect improvement with more support examples, though not guaranteed for every class given imbalance.
- Large `_std` at 1-shot is expected -- a single support example per class is a high-variance draw. Shrinking std as k grows is a good sign the setup is stable.
- If few-shot (even 5-shot) underperforms the Phase 1 linear baseline, that's a real, reportable finding: it quantifies how much a fully-trained classifier on all train data buys you over a handful of examples per class -- not a bug.
- Do **not** proceed to Phase 3 (metadata fusion) until this table is recorded.